# HalluGuard training and evaluation

This Kaggle notebook contains the fine-tuning and evaluation code. It expects the original development and test CSVs as inputs. For inference without training, use the model showcase notebook.

In [ ]:
# ============================================================
# HALLUCINATION DETECTION — DeBERTa V3 LARGE (v4 — ENHANCED)
# Complete Kaggle Training Pipeline
# ============================================================
# Major Upgrades in v4:
#   1. Model: MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli
#      - Pre-trained on FEVER (Fact Extraction & Verification) + MNLI + ANLI.
#      - Deeply understands premise-claim consistency out of the box.
#   2. Premise-First Tokenization:
#      - Formats as [CLS] Premise [SEP] Claim [SEP] to match standard NLI/FEVER
#        logic (Context A = Evidence, Context B = Claim under test).
#   3. Zero Label Smoothing (0.0):
#      - Removed smoothing so subtle factual contradictions (dates, names, quantities)
#        are strictly penalized.
#   4. Dynamic Decision Threshold Optimization:
#      - Calibrates optimal probability threshold on the validation split.
#      - Evaluates the final test set with both default (0.50) and optimal threshold.
#   5. Fixes HuggingFace deprecation warnings (dtype, warmup_steps).
# ============================================================

# -----------------------
# 1. Imports
# -----------------------
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import json
import glob
import pandas as pd
import numpy as np
import torch
from datasets import Dataset, ClassLabel
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback
)

# -----------------------
# 2. GPU Check
# -----------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("=" * 60)
print("HALLUCINATION DETECTION TRAINING (v4 — ENHANCED)")
print("=" * 60)
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print(f"GPUs visible: {torch.cuda.device_count()}")
else:
    print("WARNING: No GPU detected.")

# -----------------------
# 3. Load Data
# -----------------------
print("\nLoading CSV files...")

def find_file(filename):
    """Search for a file in common Kaggle input paths."""
    search_paths = [
        f"/kaggle/input/**/{filename}",
        f"/kaggle/input/{filename}",
    ]
    for pattern in search_paths:
        matches = glob.glob(pattern, recursive=True)
        if matches:
            return matches[0]
    raise FileNotFoundError(
        f"Could not find {filename} in /kaggle/input/. "
        f"Check uploaded datasets."
    )

train_path = find_file("train_finally_lastone_new.csv")
test_path  = find_file("final_test_file.csv")
print(f"Found train: {train_path}")
print(f"Found test:  {test_path}")

train_df = pd.read_csv(train_path)
test_df  = pd.read_csv(test_path)
if len(test_df) != 32602:
    raise ValueError(f"Expected 32,602 rows in final_test_file.csv, found {len(test_df):,}")

# -----------------------
# 4. Clean Labels & Types
# -----------------------
print("\nValidating labels and text fields...")

train_df['label'] = pd.to_numeric(train_df['label'], errors='coerce')
train_df = train_df[train_df['label'].isin([0, 1])].copy()
train_df['label'] = train_df['label'].astype(int)

test_df['label'] = pd.to_numeric(test_df['label'], errors='coerce')
test_df = test_df[test_df['label'].isin([0, 1])].copy()
test_df['label'] = test_df['label'].astype(int)

train_df['claim']   = train_df['claim'].astype(str)
train_df['premise'] = train_df['premise'].astype(str)

test_df['claim']   = test_df['claim'].astype(str)
test_df['premise'] = test_df['premise'].astype(str)

train_df = train_df.dropna(subset=['claim', 'premise', 'label']).reset_index(drop=True)
test_df  = test_df.dropna(subset=['claim', 'premise', 'label']).reset_index(drop=True)

# -----------------------
# 5. Remove Duplicates
# -----------------------
print("\nRemoving any remaining duplicates...")
train_dupes = train_df.duplicated(subset=['claim', 'premise', 'label']).sum()
test_dupes  = test_df.duplicated(subset=['claim', 'premise', 'label']).sum()
print(f"  Train duplicates found: {train_dupes}")
print(f"  Test duplicates found:  {test_dupes}")

train_df = train_df.drop_duplicates(subset=['claim', 'premise', 'label']).reset_index(drop=True)
test_df  = test_df.drop_duplicates(subset=['claim', 'premise', 'label']).reset_index(drop=True)

print(f"\nFinal Train rows: {len(train_df)}")
print(f"Final Test rows:  {len(test_df)}")
print(f"\nTrain label distribution:\n{train_df['label'].value_counts()}")
print(f"\nTest label distribution:\n{test_df['label'].value_counts()}")

# -----------------------
# 6. Class Weights
#"Hey model, if you make a mistake and 
# misclassify a 1, you will be penalized
#  almost twice as much as if you 
# misclassify a 0."
#
#
# -----------------------
label_counts    = train_df['label'].value_counts()
imbalance_ratio = label_counts.max() / label_counts.min()
print(f"\nImbalance ratio: {imbalance_ratio:.2f}:1")

n_total  = len(train_df)
n_class0 = (train_df['label'] == 0).sum()
n_class1 = (train_df['label'] == 1).sum()
weight0  = n_total / (2 * n_class0)
weight1  = n_total / (2 * n_class1)
print(f"Class weights — 0: {weight0:.3f}, 1: {weight1:.3f}")

# -----------------------
# 7. Load Tokenizer (FEVER/NLI DeBERTa Large)
# -----------------------
model_name = "MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli"
print(f"\nLoading tokenizer: {model_name}")
tokenizer = AutoTokenizer.from_pretrained(model_name)

# -----------------------
# 8. Sequence Length Analysis (Premise First)
# -----------------------
print("\nAnalyzing sequence lengths (Premise First)...")
sample_size = min(2000, len(train_df))
sample_df   = train_df.sample(n=sample_size, random_state=42)

lengths = []
for premise, claim in zip(sample_df["premise"], sample_df["claim"]):
    tokens = tokenizer(str(premise), str(claim), truncation=False)
    lengths.append(len(tokens["input_ids"]))

print(f"Sequence length stats (sampled {sample_size} examples):")
print(f"  Mean:            {np.mean(lengths):.1f}")
print(f"  Median:          {np.median(lengths):.0f}")
print(f"  95th percentile: {np.percentile(lengths, 95):.0f}")
print(f"  99th percentile: {np.percentile(lengths, 99):.0f}")
print(f"  Max:             {max(lengths)}")

max_length = 256
print(f"\nUsing max_length: {max_length} (fixed, covers 99%+ of data)")

# -----------------------
# 9. Tokenization (PREMISE FIRST: [CLS] Premise [SEP] Claim [SEP])
# -----------------------
def tokenize_function(example):
    return tokenizer(
        example["premise"],
        example["claim"],
        truncation=True,
        max_length=max_length,
        padding=False
    )

# -----------------------
# 10. Create HuggingFace Datasets
# -----------------------
print("\nCreating datasets...")
train_dataset = Dataset.from_pandas(train_df[['claim', 'premise', 'label']], preserve_index=False)
test_dataset  = Dataset.from_pandas(test_df[['claim', 'premise', 'label']], preserve_index=False)

label_feature = ClassLabel(names=['0', '1'])
train_dataset = train_dataset.cast_column('label', label_feature)
test_dataset  = test_dataset.cast_column('label', label_feature)

print("Tokenizing datasets (premise first)...")
train_dataset = train_dataset.map(tokenize_function, batched=True, batch_size=1000)
test_dataset  = test_dataset.map(tokenize_function, batched=True, batch_size=1000)

# -----------------------
# 11. Train / Validation Split
# -----------------------
print("\nSplitting train into train/validation (85/15)...")
train_val_split = train_dataset.train_test_split(
    test_size=0.15,
    seed=42,
    stratify_by_column="label"
)

print(f"  Train size:      {len(train_val_split['train']):,}")
print(f"  Validation size: {len(train_val_split['test']):,}")
print(f"  Test size:       {len(test_dataset):,}")

# Remove text columns (not needed after tokenization)
cols_to_remove = ["claim", "premise"]
for split_name in train_val_split:
    existing = [c for c in cols_to_remove if c in train_val_split[split_name].column_names]
    if existing:
        train_val_split[split_name] = train_val_split[split_name].remove_columns(existing)

existing_test = [c for c in cols_to_remove if c in test_dataset.column_names]
if existing_test:
    test_dataset = test_dataset.remove_columns(existing_test)

# -----------------------
# 12. Load Model
# -----------------------
print(f"\nLoading model: {model_name}")
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    ignore_mismatched_sizes=True,
    dtype=torch.float32
)
model = model.float()
model.to(device)
print(f"Model dtype: {next(model.parameters()).dtype}")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

# -----------------------
# 13. Weighted Trainer
# -----------------------
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None, **kwargs):
        labels   = inputs.get("labels")
        outputs  = model(**inputs)
        logits   = outputs.get("logits").float()
        weights  = torch.tensor(
            [weight0, weight1],
            dtype=torch.float32,
            device=logits.device
        )
        loss = torch.nn.CrossEntropyLoss(weight=weights)(logits, labels)
        return (loss, outputs) if return_outputs else loss

# -----------------------
# 14. Compute Metrics
# -----------------------
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    acc = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="binary", zero_division=0
    )
    _, _, f1_per_class, _ = precision_recall_fscore_support(
        labels, preds, average=None, zero_division=0
    )
    cm = confusion_matrix(labels, preds, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel() if cm.shape == (2, 2) else (0, 0, 0, 0)

    return {
        "accuracy":  round(float(acc), 4),
        "f1":        round(float(f1), 4),
        "precision": round(float(precision), 4),
        "recall":    round(float(recall), 4),
        "f1_class0": round(float(f1_per_class[0]), 4) if len(f1_per_class) > 0 else 0.0,
        "f1_class1": round(float(f1_per_class[1]), 4) if len(f1_per_class) > 1 else 0.0,
        "true_pos":  int(tp),
        "true_neg":  int(tn),
        "false_pos": int(fp),
        "false_neg": int(fn),
    }

# -----------------------
# 15. Training Arguments
# -----------------------
n_train_samples = len(train_val_split['train'])
effective_batch = 4 * 8  # 32
steps_per_epoch = n_train_samples // effective_batch
eval_steps = max(500, steps_per_epoch // 3)  # Eval ~3 times per epoch
warmup_steps = int(0.06 * steps_per_epoch * 4)  # 6% warmup across planned epochs

print(f"\nSteps per epoch: {steps_per_epoch}")
print(f"Eval every: {eval_steps} steps")
print(f"Warmup steps: {warmup_steps}")

training_args = TrainingArguments(
    output_dir="/kaggle/working/results",

    # Learning rate & schedule
    learning_rate=8e-6,
    num_train_epochs=4,
    warmup_steps=warmup_steps,
    lr_scheduler_type="cosine",

    # Batch sizes for T4 GPU (435M params)
    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=8,

    # Evaluation & checkpointing
    eval_strategy="steps",
    eval_steps=eval_steps,
    save_strategy="steps",
    save_steps=eval_steps,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,

    logging_steps=100,
    report_to="none",

    # Precision
    fp16=True,
    bf16=False,

    dataloader_num_workers=2,
    dataloader_pin_memory=True,

    # Regularization
    optim="adamw_torch",
    weight_decay=0.02,
    max_grad_norm=1.0,

    # Zero label smoothing for strict fact verification
    label_smoothing_factor=0.0,

    seed=42,
)

# -----------------------
# 16. Data Collator
# -----------------------
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    padding="longest",
    max_length=max_length
)

# -----------------------
# 17. Trainer
# -----------------------
print("\nInitializing Trainer...")
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_val_split["train"],
    eval_dataset=train_val_split["test"],
    compute_metrics=compute_metrics,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

# -----------------------
# 18. Training
# -----------------------
print("\n" + "="*60)
print("STARTING TRAINING (v4 — ENHANCED)")
print("="*60)
print(f"  Model:           {model_name}")
print(f"  Input Order:     [PREMISE, CLAIM] (Premise-first)")
print(f"  Train samples:   {len(train_val_split['train']):,}")
print(f"  Val samples:     {len(train_val_split['test']):,}")
print(f"  Test samples:    {len(test_dataset):,}")
print(f"  Effective batch: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"  Learning rate:   {training_args.learning_rate}")
print(f"  Label smoothing: {training_args.label_smoothing_factor}")
print(f"  Early stop:      patience=3 on F1")
print("="*60)

trainer.train()

print("\n" + "="*60)
print("TRAINING COMPLETE")
print("="*60)

# -----------------------
# 19. Validation Set Evaluation & Threshold Tuning
# -----------------------
print("\n" + "="*60)
print("VALIDATION & THRESHOLD OPTIMIZATION")
print("="*60)

val_preds_raw = trainer.predict(train_val_split["test"])
val_logits = val_preds_raw.predictions
val_probs = torch.softmax(torch.tensor(val_logits), dim=-1).numpy()
val_probs_class1 = val_probs[:, 1]
val_labels = val_preds_raw.label_ids

# Search for optimal threshold on validation set
best_thresh = 0.50
best_f1 = 0.0
best_acc = 0.0

for thresh in np.arange(0.20, 0.81, 0.02):
    preds = (val_probs_class1 >= thresh).astype(int)
    _, _, f1, _ = precision_recall_fscore_support(val_labels, preds, average="binary", zero_division=0)
    acc = accuracy_score(val_labels, preds)
    if f1 > best_f1:
        best_f1 = f1
        best_thresh = round(float(thresh), 2)
        best_acc = round(float(acc), 4)

default_val_preds = (val_probs_class1 >= 0.50).astype(int)
_, _, def_f1, _ = precision_recall_fscore_support(val_labels, default_val_preds, average="binary", zero_division=0)
def_acc = accuracy_score(val_labels, default_val_preds)
print(f"Validation Tuning Results:")
print(f"  Default Threshold (0.50) -> Accuracy: {def_acc:.4f} | F1: {def_f1:.4f}")
print(f"  Optimal Threshold ({best_thresh:.2f}) -> Accuracy: {best_acc:.4f} | F1: {best_f1:.4f}")

# -----------------------
# 20. Final Test Set Evaluation (Dual: Default vs Optimal Threshold)
# -----------------------
print("\n" + "="*60)
print("FINAL TEST SET EVALUATION")
print("="*60)

test_preds_raw = trainer.predict(test_dataset)
test_logits = test_preds_raw.predictions
test_probs = torch.softmax(torch.tensor(test_logits), dim=-1).numpy()
test_probs_class1 = test_probs[:, 1]
test_labels = test_preds_raw.label_ids

def print_eval_metrics(title, preds, labels):
    acc = accuracy_score(labels, preds)
    prec, rec, f1, _ = precision_recall_fscore_support(labels, preds, average="binary", zero_division=0)
    _, _, f1_pc, _ = precision_recall_fscore_support(labels, preds, average=None, zero_division=0)
    cm = confusion_matrix(labels, preds, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel() if cm.shape == (2, 2) else (0, 0, 0, 0)
    print(f"\n{title}:")
    print(f"  Accuracy:    {acc:.4f}")
    print(f"  F1 (Binary): {f1:.4f}")
    print(f"  Precision:   {prec:.4f}")
    print(f"  Recall:      {rec:.4f}")
    print(f"  F1 Class 0:  {f1_pc[0]:.4f}")
    print(f"  F1 Class 1:  {f1_pc[1]:.4f}")
    print(f"  TP: {tp} | TN: {tn} | FP: {fp} | FN: {fn}")

# Default threshold (0.50)
default_test_preds = (test_probs_class1 >= 0.50).astype(int)
print_eval_metrics("Results with Default Threshold (0.50)", default_test_preds, test_labels)

# Tuned threshold
tuned_test_preds = (test_probs_class1 >= best_thresh).astype(int)
print_eval_metrics(f"Results with Tuned Threshold ({best_thresh:.2f})", tuned_test_preds, test_labels)

# -----------------------
# 21. Save Model & Threshold Config
# -----------------------
output_path = "/kaggle/working/hallucination_deberta_v3_v4"
os.makedirs(output_path, exist_ok=True)
trainer.save_model(output_path)
tokenizer.save_pretrained(output_path)

# Save optimal threshold config
thresh_config = {
    "model_name": model_name,
    "optimal_threshold": float(best_thresh),
    "input_format": "[CLS] Premise [SEP] Claim [SEP]",
    "max_length": max_length
}
with open(os.path.join(output_path, "threshold_config.json"), "w") as f:
    json.dump(thresh_config, f, indent=2)

print(f"\nModel and threshold config saved to: {output_path}")

model_size = sum(
    os.path.getsize(os.path.join(output_path, f))
    for f in os.listdir(output_path)
    if os.path.isfile(os.path.join(output_path, f))
)
print(f"Total model size: {model_size / 1e6:.2f} MB")

# -----------------------
# 22. Quick Inference Test (Premise-First + Optimal Threshold)
# -----------------------
print("\n" + "="*60)
print(f"QUICK INFERENCE TEST (Using Threshold: {best_thresh:.2f})")
print("="*60)

model.eval()
test_samples = [
    {
        "claim": "The French Revolution began in 1789.",
        "premise": "The French Revolution was a period of radical political and societal change in France that began with the Estates General of 1789.",
        "expected": 1
    },
    {
        "claim": "Napoleon was defeated at Waterloo in 1820.",
        "premise": "Napoleon was finally defeated at Waterloo in 1815.",
        "expected": 0
    },
    {
        "claim": "The Earth orbits the Sun once every 365.25 days.",
        "premise": "Earth takes approximately 365.25 days to complete one orbit around the Sun, which is why we have a leap year every four years.",
        "expected": 1
    },
    {
        "claim": "Albert Einstein discovered penicillin in 1928.",
        "premise": "Penicillin was discovered by Alexander Fleming in 1928 at St. Mary's Hospital in London.",
        "expected": 0
    },
]

correct = 0
for sample in test_samples:
    # Premise first, claim second
    inputs = tokenizer(
        sample["premise"],
        sample["claim"],
        return_tensors="pt",
        truncation=True,
        max_length=max_length,
        padding=True
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1)[0]
        prob_class1 = probs[1].item()
        pred = 1 if prob_class1 >= best_thresh else 0

    is_correct = pred == sample['expected']
    correct += int(is_correct)
    print(f"  Premise:  {sample['premise'][:80]}...")
    print(f"  Claim:    {sample['claim']}")
    print(f"  Expected: {sample['expected']} | Predicted: {pred} | Prob(1): {prob_class1:.3f} (Thresh: {best_thresh:.2f})")
    print(f"  {'CORRECT' if is_correct else 'WRONG'}")
    print()

print(f"Inference test: {correct}/{len(test_samples)} correct")
print("="*60)
print("PIPELINE COMPLETE (v4)")
print("="*60)